In [14]:
!pip install groq

In [15]:
from groq import Groq
from google.colab import userdata
import pandas as pd
import json

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

MODEL = "openai/gpt-oss-120b"

print("NutriGuide AI connected!")

NutriGuide AI connected!


In [16]:
food_data = {
    "food": [
        "boiled egg",
        "milk",
        "banana",
        "rice",
        "chicken breast"
    ],
    "serving_g": [50, 100, 100, 100, 100],
    "calories": [78, 61, 89, 130, 165],
    "protein": [6.3, 3.2, 1.1, 2.7, 31],
    "carbs": [0.6, 4.8, 22.8, 28, 0],
    "fat": [5.3, 3.3, 0.3, 0.3, 3.6]
}

food_df = pd.DataFrame(food_data)

food_df

,food,serving_g,calories,protein,carbs,fat
0,boiled egg,50,78,6.3,0.6,5.3
1,milk,100,61,3.2,4.8,3.3
2,banana,100,89,1.1,22.8,0.3
3,rice,100,130,2.7,28.0,0.3
4,chicken breast,100,165,31.0,0.0,3.6


In [17]:
def get_food_nutrition(food_name):

    result = food_df[
        food_df["food"].str.lower() == food_name.lower()
    ]

    if result.empty:
        return {"error": f"{food_name} not found in database"}

    food = result.iloc[0]

    return {
        "food": food["food"],
        "serving_g": int(food["serving_g"]),
        "calories": float(food["calories"]),
        "protein": float(food["protein"]),
        "carbs": float(food["carbs"]),
        "fat": float(food["fat"])
    }

In [18]:
get_food_nutrition("banana")

{'food': 'banana',
 'serving_g': 100,
 'calories': 89.0,
 'protein': 1.1,
 'carbs': 22.8,
 'fat': 0.3}

In [19]:
conversation_history = [
    {
        "role": "system",
        "content": """
        You are NutriGuide AI, an intelligent Indian food nutrition assistant.

        Your responsibilities:
        1. Answer food nutrition questions.
        2. Retrieve nutrition values using the available database.
        3. Help users compare foods and understand nutrients.
        4. Maintain awareness of previous conversation messages.

        Context rules:
        - Use previous messages to understand the user.
        - Remember information shared earlier in this conversation.
        - Do not ask the user to repeat information already available.
        - Never invent nutrition values.
        - Be friendly and explain answers simply.
        """
    }
]

print("Conversation memory initialized!")

Conversation memory initialized!


In [28]:

def chat_with_agent(user_input):

    # 1. Add user message to conversation history
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # 2. Send conversation history and tools to Groq
    response = client.chat.completions.create(
        model=MODEL,
        messages=conversation_history,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # 3. Check whether AI requested a tool
    if message.tool_calls:

        conversation_history.append(message)

        # 4. Execute the requested tool
        for tool_call in message.tool_calls:

            function_name = tool_call.function.name

            arguments = json.loads(
                tool_call.function.arguments
            )

            if function_name == "get_food_nutrition":

                result = get_food_nutrition(**arguments)

                print("TOOL EXECUTED:", result)

                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        # 5. Ask AI to generate the final answer
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=conversation_history,
            tools=tools
        )

        ai_response = final_response.choices[0].message.content

    else:
        ai_response = message.content

    # 6. Store AI response in conversation history
    conversation_history.append({
        "role": "assistant",
        "content": ai_response
    })

    return ai_response


In [21]:
print(chat_with_agent("Hi! My name is Aswini. I am an ECE graduate."))

Hello Aswini! 👋 Nice to meet you. I'm NutriGuide, your Indian‑food nutrition assistant. How can I help you today? Whether you’re curious about the calories in a dish, the protein content of your favorite snack, or want to compare two foods, just let me know!


In [22]:
print(chat_with_agent("What is my name and educational background?"))

Your name is **Aswini**, and you mentioned that you’re an **ECE (Electronics and Communication Engineering) graduate**. Let me know if there’s anything else you’d like to discuss!


In [23]:
print(chat_with_agent(
    "Since you know my name, greet me and tell me how much protein is in 100g of banana."
))

Hi Aswini! 👋

The protein content in **100 g of raw banana (peeled)** is approximately **1.1 g**. If you need more details—like calories, carbs, or how bananas compare with other fruits—just let me know!


In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_food_nutrition",
            "description": "Retrieve nutrition values from the food database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "food_name": {
                        "type": "string",
                        "description": "Name of the food"
                    }
                },
                "required": ["food_name"]
            }
        }
    }
]

print("Nutrition tool registered!")

Nutrition tool registered!


In [26]:
print(chat_with_agent("Hi! My name is Aswini. I am an ECE graduate."))

Hi Aswini! 👋 Great to see you again.

Just to confirm, you’re **Aswini**, an **ECE graduate**.  

**Protein in 100 g of raw banana (peeled):** **≈ 1.1 g**.

If you’d like to know about other nutrients in bananas or compare them with another food, just let me know!


In [29]:
print(chat_with_agent(
    "How much protein is in 100g of banana?"
))

In 100 g of raw banana (peeled), there’s about **1.1 grams of protein**. If you’d like the full nutrition breakdown (calories, carbs, vitamins, etc.) or want to compare bananas with another fruit, just let me know!


In [30]:
print("Last user message:")
print(conversation_history[-2])

print("\nLast assistant message:")
print(conversation_history[-1])

Last user message:
{'role': 'user', 'content': 'How much protein is in 100g of banana?'}

Last assistant message:
{'role': 'assistant', 'content': 'In\u202f100\u202fg of raw banana (peeled), there’s about **1.1\u202fgrams of protein**. If you’d like the full nutrition breakdown (calories, carbs, vitamins, etc.) or want to compare bananas with another fruit, just let me know!'}


In [31]:
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You must use the get_food_nutrition tool to answer nutrition questions."
        },
        {
            "role": "user",
            "content": "How much protein is in 100g of banana?"
        }
    ],
    tools=tools,
    tool_choice="required"
)

print(test_response.choices[0].message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer using get_food_nutrition. Let\'s call function with food_name "banana".', tool_calls=[ChatCompletionMessageToolCall(id='fc_f05fee1e-824c-4f1a-ab11-1111f00b29c7', function=Function(arguments='{"food_name":"banana"}', name='get_food_nutrition'), type='function')])


In [32]:
conversation_history[0]["content"] = """
You are NutriGuide AI, an intelligent Indian food nutrition assistant.

Rules:
1. For every question asking about food nutrition values,
   you MUST call the get_food_nutrition tool.
2. Never invent nutrition values from your own knowledge.
3. If a food is not available in the database, clearly say so.
4. Use previous conversation messages to understand the user.
5. Be friendly and explain answers simply.
"""

print("System instructions updated!")

System instructions updated!


In [33]:
print(chat_with_agent(
    "How much protein is in 100g of banana?"
))

TOOL EXECUTED: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}
Hi Aswini! 👋

According to the nutrition data:

- **Food:** Banana (raw, peeled)  
- **Serving size:** 100 g  
- **Protein:** **1.1 g**

If you’d like more details—like calories, carbs, or vitamins—for this serving, just let me know!


In [34]:
print(chat_with_agent(
    "What is my name?"
))

Your name is **Aswini**.
